# Advanced tutorial: Compare sensor and grab-sample measurements

## Research question

**How closely do daily continuous-sensor nitrate measurements agree with laboratory grab samples collected at the same station?**

## `stream_utils` functions used

- `StreamClient.find_sites()`
- `StreamClient.read_water_quality()`
- `StreamClient.read_grab_samples()`
- `common_observation_period()`
- `resample_timeseries()`
- `join_timeseries()`

In [ ]:
import pandas as pd

from stream_utils.client import StreamClient
from stream_utils.joins import join_timeseries
from stream_utils.temporal import common_observation_period, resample_timeseries

stream = StreamClient()

## 1. Find stations with both nitrate datasets

In [ ]:
sites = stream.find_sites(
    variables="NO3_mgNL",
    grab_variables="NO3_mgNL",
)
print(f"Found {len(sites):,} candidate stations")
sites.head()

## 2. Find a station with usable records in both datasets

Metadata availability does not guarantee that a station has non-null approved sensor values or measurements on the same days as its grab samples. Read the two nitrate columns for the candidate stations, then choose the station with the most matched sampling days.

In [ ]:
candidate_ids = sites["STREAM_ID"].tolist()
all_sensor = stream.read_water_quality(
    candidate_ids, "NO3_mgNL", quality="approved"
).dropna(subset=["DateTime", "NO3_mgNL"])
all_grab = stream.read_grab_samples(
    candidate_ids, "NO3_mgNL"
).dropna(subset=["DateTime", "NO3_mgNL"])

sensor_days = all_sensor[["STREAM_ID", "DateTime"]].copy()
grab_days = all_grab[["STREAM_ID", "DateTime"]].copy()
sensor_days["DateTime"] = pd.to_datetime(sensor_days["DateTime"])
grab_days["DateTime"] = pd.to_datetime(grab_days["DateTime"])
sensor_days["day"] = sensor_days["DateTime"].dt.normalize()
grab_days["day"] = grab_days["DateTime"].dt.normalize()
matched_days = (
    sensor_days[["STREAM_ID", "day"]].drop_duplicates()
    .merge(grab_days[["STREAM_ID", "day"]].drop_duplicates(),
           on=["STREAM_ID", "day"], how="inner")
    .groupby("STREAM_ID").size().sort_values(ascending=False)
)
if matched_days.empty:
    raise ValueError("No station has approved sensor and grab-sample nitrate on the same day")

example_site = matched_days.index[0]
sensor = all_sensor[all_sensor["STREAM_ID"] == example_site].copy()
grab = all_grab[all_grab["STREAM_ID"] == example_site].copy()
common_start, common_end = common_observation_period(sensor, grab)
print(f"Selected {example_site} with {matched_days.iloc[0]} matched day(s)")
print(f"Common observation period: {common_start} to {common_end}")

## 3. Compare measurements by sampling day

Exact hourly timestamps are unlikely to match. Converting both datasets to daily values makes the comparison explicit: the sensor value is the daily mean, while multiple grab samples on one day are averaged.

In [ ]:
daily_sensor = resample_timeseries(sensor, "NO3_mgNL", "daily", "mean")
daily_grab = resample_timeseries(grab, "NO3_mgNL", "daily", "mean")
comparison = join_timeseries(
    daily_sensor, daily_grab, suffixes=("_sensor", "_grab")
)
comparison["difference_mgNL"] = (
    comparison["NO3_mgNL_sensor"] - comparison["NO3_mgNL_grab"]
)
comparison.head()

In [ ]:
comparison[["NO3_mgNL_sensor", "NO3_mgNL_grab", "difference_mgNL"]].describe()